In [1]:
import sys
import os

# Add the local stonesoup directory to sys.path
project_path = r"C:\Users\joesb\Documents\stonesoup"  # Adjust this to your actual path
if project_path not in sys.path:
    sys.path.insert(0, project_path)

# print(sys.path)


In [2]:
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.base_driver import GaussianResidualApproxCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levylinear import LevyLangevin, CombinedLinearLevyTransitionModel

import numpy as np
from datetime import datetime, timedelta

# And the clock starts
start_time = datetime.now().replace(microsecond=0)

seed = 1 # Random seem for reproducibility

# Driving process parameters
mu_W = 0
sigma_W2 = 4
alpha = 1.4
c=10
noise_case=GaussianResidualApproxCase()

# Model parameters
theta=0.15

driver_x = AlphaStableNSMDriver(mu_W=mu_W, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, noise_case=noise_case)
driver_y = driver_x # Same driving process in both dimensions and sharing the same latents (jumps)
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta, mu_W=-0.02)
langevin_y = LevyLangevin(driver=driver_y, damping_coeff=theta)
transition_model = CombinedLinearLevyTransitionModel([langevin_x, langevin_y])

In [3]:
timesteps = [start_time]

truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])

num_steps = 10
for k in range(num_steps):
    timesteps.append(start_time+timedelta(seconds=1*(k+1)))  # add next timestep to list of timesteps
    truth.append(GroundTruthState(
        transition_model.function(truth[k], noise=True, time_interval=timedelta(seconds=1)),
        timestamp=timesteps[k+1]))

In [4]:
from stonesoup.types.detection import Detection
from stonesoup.models.measurement.linear import LinearGaussian

measurement_model = LinearGaussian(
    ndim_state=4,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 2),  # Mapping measurement vector index to state index
    noise_covar=np.array([[15, 0],  # Covariance matrix for Gaussian PDF
                          [0, 15]])
    )

measurements = []
for state in truth:
    measurement = measurement_model.function(state, noise=True)
    measurements.append(Detection(measurement,
                                  timestamp=state.timestamp,
                                  measurement_model=measurement_model))

In [5]:
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler 
from stonesoup.updater.particle import MarginalisedParticleUpdater

predictor = MarginalisedParticlePredictor(transition_model=transition_model)
resampler = SystematicResampler()
updater = MarginalisedParticleUpdater(measurement_model, resampler)

In [6]:
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import StateVectors

number_particles = 5

# Sample from the prior Gaussian distribution
states = multivariate_normal.rvs(np.array([0, 1, 0, 1]),
                                  np.diag([1., 1., 1., 1.]),
                                  size=number_particles)
covars = np.stack([np.eye(4) * 100 for i in range(number_particles)], axis=2) # (M, M, N)

# Create prior particle state.
prior = MarginalisedParticleState(
    state_vector=StateVectors(states.T),
    covariance=covars,
    weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time-timedelta(seconds=1))

In [ ]:
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

track = Track()


for measurement in measurements:
    prediction = predictor.predict(prior, timestamp=measurement.timestamp)
    hypothesis = SingleHypothesis(prediction, measurement)
    post = updater.update(hypothesis, store_extra_data=True)
    track.append(post)
    prior = track[-1]
    print(f"track length ={len(track)} of {len(measurements)}")

In [8]:
from stonesoup.smoother.particle import MarginalisedKalmanSmoother, ParticleSmoother, DescendantSmoother
particlesmoother=ParticleSmoother(track=track)
conditionalsmoother=MarginalisedKalmanSmoother(track=track)

In [10]:
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
axis_label_list=["x","dx_dt","y","dy_dt"]
particle_plotter_dict = {}
for i,label in enumerate(axis_label_list):
    particle_plotter_dict[label]= Plotterly(autosize=False, width=1200,height=600, dimension=Dimension.ONE, axis_labels=[label])
    particle_plotter_dict[label].plot_ground_truths(truth, [i],mode="lines", line=dict(width=1))
    if label =="x" or label=="y":
        particle_plotter_dict[label].plot_measurements(measurements, [i],marker=dict(symbol="x",size=4))
    particle_plotter_dict[label].plot_tracks(track, [i],uncertainty=True,particle=True,mode="lines", track_label="Original Track",line=dict(width=1))
    file_path = Path(rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\TimeVaryingMeanPlots\{num_steps}steps_{number_particles}p\1D_plot_{label}.html")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    particle_plotter_dict[label].fig.write_html(str(file_path))
    # particle_plotter_dict[label].fig.show()